# Pixel assignment of the injected GCs

Takes the GC coordinates from `gc_injected_data.txt` (written by
`rubix_pipeline_single_function_shard_map_fits_gsf_emiles_hub26_only_gcs_no_pse_no_noise_june26.ipynb`)
and returns the spaxel each GC is assigned to, using the same rubix functions as the
`spaxel_assignment` step of the pipeline (`get_spatial_bin_edges` + `square_spaxel_assignment`).

Index conventions (0-based):
- `pixel_assignment = ix + n_pix * iy` (same as `rubixdata.stars.pixel_assignment`), `ix` along x, `iy` along y
- rubix cube in the mock notebook: spectrum of a GC = `rubixdata[iy, ix, :]`
- FITS file written by `store_fits` (it stores `datacube.T`, so x and y are swapped):
  `hdul["DATA"].data[:, ix, iy]`, i.e. in DS9 the GC sits at pixel X = iy + 1, Y = ix + 1

In [6]:
import os
import sys

os.environ.setdefault("JAX_PLATFORMS", "cpu")  # tiny calculation, no GPU needed

# use the rubix code of this repository (the code that produced the mock; it knows MUSE_HUB26)
repo_root = os.path.abspath("..")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import jax.numpy as jnp
import numpy as np
import pandas as pd
import rubix
from rubix.core.telescope import get_spatial_bin_edges, get_telescope
from rubix.telescope.utils import mask_particles_outside_aperture, square_spaxel_assignment

print("rubix imported from", os.path.dirname(rubix.__file__))

rubix imported from /home/annalena/rubix/rubix


In [7]:
# GC table written by the mock notebook
run_dir = "./output/ngc1399/psf_noise"
gc_file = f"{run_dir}/gc_injected_data.txt"

# same settings as in the mock notebook
telescope_name = "MUSE_HUB26"  # use "MUSE_WFM" for ./output/hub2026/gcs_200_no_psf_no_noise
dist_z = 0.003
cosmology_name = "PLANCK15"

config = {
    "logger": {
        "log_level": "WARNING",
        "log_file_path": None,
        "format": "%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    },
    "telescope": {"name": telescope_name},
    "galaxy": {"dist_z": dist_z},
    "cosmology": {"name": cosmology_name},
}

In [8]:
gc_data = np.loadtxt(gc_file)  # columns: age metallicity mass vx vy vz cx cy cz
coords = jnp.array(gc_data[:, 6:9])  # kpc, float32 like inputdata.stars.coords in the mock notebook
n_gc = coords.shape[0]
print(f"{n_gc} GCs loaded from {gc_file}")

200 GCs loaded from ./output/ngc1399/psf_noise/gc_injected_data.txt


In [9]:
n_pix = int(get_telescope(config).sbin)  # spaxels per side
spatial_bin_edges = get_spatial_bin_edges(config)  # kpc, the same edges the pipeline uses

pixel_assignment = np.asarray(square_spaxel_assignment(coords, spatial_bin_edges))
ix = pixel_assignment % n_pix  # x index
iy = pixel_assignment // n_pix  # y index

inside = np.asarray(mask_particles_outside_aperture(coords, spatial_bin_edges))
print(f"{telescope_name}: {n_pix} x {n_pix} spaxels, "
      f"FOV {float(spatial_bin_edges[0]):.4f} ... {float(spatial_bin_edges[-1]):.4f} kpc")
print(f"{inside.sum()} / {n_gc} GCs inside the FOV (rubix puts GCs outside onto the edge spaxels)")
print(f"{len(np.unique(pixel_assignment))} different spaxels")

2026-11-20 14:19:42,033 - rubix - INFO - No telescope config provided, falling back to /home/annalena/rubix/rubix/telescope/telescopes.yaml
/home/annalena/rubix/rubix/telescope/factory.py:32: UserWarning: No telescope config provided, using default stored in /home/annalena/rubix/rubix/telescope/telescopes.yaml
  warnings.warn(
2026-11-20 14:19:42,049 - rubix - INFO - No telescope config provided, falling back to /home/annalena/rubix/rubix/telescope/telescopes.yaml


MUSE_HUB26: 330 x 330 spaxels, FOV -2.1163 ... 2.1163 kpc
118 / 200 GCs inside the FOV (rubix puts GCs outside onto the edge spaxels)
184 different spaxels


In [10]:
gc_pixels = pd.DataFrame({
    "gc_id": np.arange(n_gc),  # row in gc_injected_data.txt
    "x_kpc": gc_data[:, 6],
    "y_kpc": gc_data[:, 7],
    "pixel_assignment": pixel_assignment,
    "ix": ix,
    "iy": iy,
})

out_file = f"{run_dir}/gc_pixel_assignment_{telescope_name}.txt"
np.savetxt(
    out_file,
    gc_pixels.to_numpy(),
    fmt=["%d", "%.6e", "%.6e", "%d", "%d", "%d"],
    header=" ".join(gc_pixels.columns),
    comments="# ",
)
print("saved to", out_file)
gc_pixels

saved to ./output/ngc1399/psf_noise/gc_pixel_assignment_MUSE_HUB26.txt


,gc_id,x_kpc,y_kpc,pixel_assignment,ix,iy
0,0,1.717448,1.229071,86098,298,260
1,1,-0.878224,2.845592,108666,96,329
2,2,2.425799,-2.274963,329,329,0
3,3,2.438370,-2.635642,329,329,0
4,4,-1.670842,-2.581454,34,34,0
...,...,...,...,...,...,...
195,195,2.041206,2.059720,107574,324,325
196,196,0.304311,-2.940563,188,188,0
197,197,0.704395,-0.918701,30909,219,93
198,198,-1.583864,-1.109015,25781,41,78
